# A minimal working example for generating Myerson explanations with the chemical datasets from GraphXAI

* uses the trained model in `formal/realworld/<dataset_name>/model_weights/GIN_<dataset_name>.pth` (retraining is possible with the scripts under`formal/realworld/<dataset_name>`)
* 

## Installation instructions

```sh
# updated requirements.txt
# installing dependencies (python 3.13.5)
uv init
uv venv 
source .venv/bin/activate
uv pip install -e .
uv pip install torch # torch-cluster depends on torch
uv pip install -r requirements.txt --no-build-isolation
uv pip install git+https://github.com/SamuelHomberg/myerson.git@feature
```

## Add missing EXPS folders

The calculated explanations are saved, so if the method is changed the directories have to be emptied manually. 

```sh
mkdir -p formal/realworld/benzene/EXPS/MYX
mkdir -p formal/realworld/fc/EXPS/MYX
mkdir -p formal/realworld/mutag/EXPS/MYX
# mkdir -p formal/realworld/mutag/EXPS/<all_other_explainers> # necessary for other explainers
# ...
```

In [ ]:
# imports
import os
import torch
import random
import numpy as np
from tqdm import tqdm

from formal.realworld.utils import get_model, get_exp_method, get_dataset

from graphxai.explainers import PGExplainer, MyersonExplainer_
from graphxai.utils.performance.load_exp import exp_exists_graph
from graphxai.metrics.metrics_graph import graph_exp_acc_graph, graph_exp_faith_graph


In [ ]:
# starting arguments
dataset_name = 'benzene' # ['mutag', 'fc', 'benzene']
seed_value = 912
exp_method_name = 'myx' # ['ig', 'gnnex', 'grad', ...]
model_name = 'GIN'

exp_loc = os.path.join('formal','realworld', dataset_name, 'EXPS', exp_method_name.upper())

In [ ]:
# set seeds
random.seed(seed_value)
np.random.seed(seed_value)
torch.manual_seed(seed_value)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Get dataset:
dataset = get_dataset(dataset_name, device = device)
test_inds = dataset.test_index # Index of testing objects

# Set up model:
model = get_model(model_name)

# Construct path to model:
mpath = os.path.join('formal', 'realworld', dataset_name, 'model_weights',
                     '{}_{}.pth'.format(model_name.upper(), dataset_name))
# Load model:
model.load_state_dict(torch.load(mpath))

model = model.to(device)

criterion = torch.nn.CrossEntropyLoss().to(device)


## Calculate a single explanation and GEA / GEF for Myerson

In [ ]:
# get example
example_index = random.choice(test_inds)

data, gt_exp = dataset[example_index]
data = data.to(device)

# sanity check
add_args = {'batch': torch.zeros((1,)).long().to(device)} # add batch arguments for single predictions
pred_class = model(data.x.to(device), data.edge_index.to(device), **add_args).reshape(-1, 1).argmax(dim=0)
assert pred_class == data.y.item(), f"Prediction is false"

# generate explanation
explainer = MyersonExplainer_(model)
forward_kwargs={'x': data.x.to(device),
                'label': data.y.to(device),
                'edge_index': data.edge_index.to(device)}
exp = explainer.get_explanation_graph(**forward_kwargs)

# calculate metrics
_, node_gea, edge_gea = graph_exp_acc_graph(gt_exp, exp, node_thresh_factor = 0.5)
_, node_gef, edge_gef = graph_exp_faith_graph(exp, data, model, forward_kwargs = add_args)

## Get all explanations

This is basically the script from `formal/realworld/graph_eval.py`

In [ ]:
gea_node = []
gea_edge = []

gef_node = []
gef_edge = []

add_args = {
    'batch': torch.zeros((1,)).long().to(device)
}


if exp_method_name.lower()=='pgex':
    # Train the PGExplainer
    masked_dataset = [dataset[i][0] for i in dataset.train_index] # Mask to list of only train data
    # fixed for 3-layer conv:
    explainer = PGExplainer(model, explain_graph = True, emb_layer_name='conv3', max_epochs=10, lr=0.1)
    explainer.train_explanation_model(masked_dataset, forward_kwargs=add_args) # Train model on training data


for i, idx in enumerate(tqdm(test_inds)):

    # Gets graph, ground truth explanation:
    data, gt_exp = dataset[idx]
    data = data.to(device)
    pred_class = model(data.x.to(device), data.edge_index.to(device), **add_args).reshape(-1, 1).argmax(dim=0)
    # skip if prediction is wrong
    if pred_class != data.y.item():
        continue

    # Try to load from existing dir:
    exp = exp_exists_graph(idx, path = exp_loc, get_exp = True)
    if exp is None or (exp_method_name.lower() == 'pgex'): # Don't allow PGEX to re-train on previous explanations
        if (exp_method_name.lower() == 'pgex'):
            # Explainer is set before loop
            forward_kwargs = {
                'x': data.x.to(device),
                'edge_index': data.edge_index.to(device),
                'label': pred_class,
                'forward_kwargs': add_args
            }
        else:
            explainer, forward_kwargs = get_exp_method(exp_method_name, model, criterion, pred_class, data, device)
        # generate and store explanation
        exp = explainer.get_explanation_graph(**forward_kwargs)
        torch.save(exp, open(os.path.join(exp_loc, 'exp_{:0>5d}.pt'.format(idx)), 'wb'))

    # Accuracy:
    _, node_acc, edge_acc = graph_exp_acc_graph(gt_exp, exp, node_thresh_factor = 0.5)

    gea_node.append(node_acc)
    gea_edge.append(edge_acc)

    # Faithfulness:
    _, node_faith, edge_faith = graph_exp_faith_graph(exp, data, model, forward_kwargs = add_args)

    gef_node.append(node_faith)
    gef_edge.append(edge_faith)
    if i == 3:
        break


In [ ]:
gea_node